# Ranking provisional de familias FP para el piloto

## tl;dr

Este cuaderno perfila el snapshot de ofertas ECYL direccionado por el manifiesto y construye dos lecturas separadas: sector observado y ofertas con indicios textuales razonablemente abordables desde FP. La clasificación es determinista y provisional; no sustituye una CNO oficial, que no está presente en el dataset.

## Context & Methods

### Key Assumptions

- Unidad: oferta única por `id`, usando el recurso señalado por `public/data/v1/manifest.json`.
- Ventana de captura disponible: del 4 de febrero al 3 de agosto de 2026; el corte reciente de tres meses se muestra como composición, no como prueba de tendencia.
- `sector_candidate` es una agrupación léxica amplia.
- `fp_candidate_family` exige términos más concretos y excluye profesiones universitarias o dependientes principalmente de permisos.
- `Sin asignación segura` permanece visible y no se reparte proporcionalmente.

In [1]:
from pathlib import Path
import json
import re
import unicodedata
import pandas as pd
from IPython.display import display

repo = Path.cwd()
if not (repo / 'public/data/v1/manifest.json').is_file():
    repo = repo.parent
assert (repo / 'public/data/v1/manifest.json').is_file(), repo
manifest = json.loads((repo / 'public/data/v1/manifest.json').read_text(encoding='utf-8'))
resource_path = manifest['resourceSnapshots']['jobOffers']['resourcePath'].lstrip('/')
source_path = repo / 'public' / resource_path
offers = json.loads(source_path.read_text(encoding='utf-8'))
df = pd.DataFrame(offers)
df['publishedAt'] = pd.to_datetime(df['publishedAt'], utc=True)
df['title_normalized'] = (df['title'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('ascii').str.lower())
snapshot = manifest['resourceSnapshots']['jobOffers']
print({'source_path': str(source_path), 'manifest_rows': snapshot['recordCount'], 'loaded_rows': len(df), 'manifest_generated_at': manifest['generatedAt'], 'source_updated_at': snapshot.get('sourceUpdatedAt')})

{'source_path': 'F:\\castilla_leon_rev2\\.worktrees\\salida-cyl-development\\public\\data\\v1\\snapshots\\20260809014318761-5b22c488ce4b\\job-offers.json', 'manifest_rows': 1077, 'loaded_rows': 1077, 'manifest_generated_at': '2026-08-09T01:43:18.761Z', 'source_updated_at': '2026-08-07T00:00:00.000Z'}


## Data

### 1. Perfil y controles de calidad

In [2]:
max_date = df['publishedAt'].max()
cutoff_3m = max_date - pd.DateOffset(months=3)
cutoff_6m = max_date - pd.DateOffset(months=6)
profile = pd.DataFrame([{
    'offers': len(df),
    'unique_ids': df['id'].nunique(),
    'unique_urls': df['originalUrl'].nunique(),
    'min_published': df['publishedAt'].min().date().isoformat(),
    'max_published': max_date.date().isoformat(),
    'last_3_months': int((df['publishedAt'] >= cutoff_3m).sum()),
    'last_6_months': int((df['publishedAt'] >= cutoff_6m).sum()),
    'missing_title': int(df['title'].isna().sum()),
    'missing_province': int(df['province'].isna().sum()),
    'missing_locality': int(df['locality'].isna().sum()),
}])
display(profile)
display(df.groupby(df['publishedAt'].dt.strftime('%Y-%m')).size().rename('offers').reset_index(name='offers'))

,offers,unique_ids,unique_urls,min_published,max_published,last_3_months,last_6_months,missing_title,missing_province,missing_locality
0,1077,1077,1077,2026-02-10,2026-08-07,1006,1077,0,0,84


,publishedAt,offers
0,2026-02,5
1,2026-03,21
2,2026-04,28
3,2026-05,103
4,2026-06,111
5,2026-07,563
6,2026-08,246


### 2. Clasificación léxica reproducible

Las reglas se aplican en orden y buscan términos profesionales explícitos. El resultado sirve para priorizar investigación; no aprueba relaciones ciclo–ocupación.

In [3]:
sector_rules = [
    ('Sanidad', r'enfermer|healthcare assistants|medic[oa]|fisioter|terapeutas? ocupacional|sanitari|audioprotes|farmac|higien|laboratorio clin|odont'),
    ('Servicios Socioculturales y a la Comunidad', r'asistent.*domic|ayuda a domicilio|cuidador|dependencia|discapacidad|trabajador.*social|educador.*social|tecnic.*educacion infantil|animador.*sociocultural|integrador.*social|servicios sociales|emplead.*hogar'),
    ('Hostelería y Turismo', r'cociner|camarer|pinche.*cocina|hosteler|hotel|alojamiento|recepcionista.*hotel|restauracion'),
    ('Transporte y Mantenimiento de Vehículos', r'conductor|mecanic.*automoc|chapista|vehicul|lavacoches|jefe.*trafico|autobus|camion'),
    ('Edificación y Obra Civil', r'albanil|encofrador|obra civil|obra de edificacion|pintor|empapelador|construccion|topograf|operador.*movimiento de tierras|grua torre'),
    ('Comercio y Marketing', r'comercial|dependient|cajer|reponedor|almacen|carga y descarga|carretiller|compras|marketing|atencion al cliente|expendedor|personal.*supermercado'),
    ('Agraria', r'agricol|ganader|forestal|jardiner|vid\b|vitic|ovino|caprino'),
    ('Instalación y Mantenimiento', r'electromecanic.*industrial|mantenimiento industrial|frigor|fontaner|climatiza|calefaccion|mantenimiento de equipos'),
    ('Fabricación Mecánica', r'soldador|tornero|fresador|calderer|mecaniz|estructuras metalicas'),
    ('Administración y Gestión', r'administrativ|contabilidad|fiscal|tributari|gestion ciudadana|secretari'),
    ('Industrias Alimentarias', r'matarife|carnicer|charcuter|bodeguer|industria.*alimentacion|elaboracion.*aliment|produccion.*aliment'),
    ('Madera, Mueble y Corcho', r'carpinter.*madera|ebanist|montador.*mueble'),
    ('Imagen Personal', r'peluquer|estetic|maquill'),
    ('Electricidad y Electrónica', r'electricist|electricidad|electroni|telecomunic'),
    ('Informática y Comunicaciones', r'programador|developer|desarrollador.*web|informatic|sistemas informatic|software|applications developer'),
    ('Actividades Físicas y Deportivas', r'animador.*deport|monitor.*deport|socorrista|acondicionamiento fisico'),
    ('Seguridad y Medio Ambiente', r'seguridad y salud|prevencion.*riesgos|bombero|emergencias'),
    ('Artes Gráficas', r'impresor|preimpres|artes graficas|disenador grafico'),
    ('Imagen y Sonido', r'audiovisual|sonido|fotograf|produccion.*audiovisual'),
    ('Química', r'laboratorio quim|quimic|control de calidad'),
    ('Textil, Confección y Piel', r'costurer|confeccion|textil|tapicer'),
    ('Energía y Agua', r'energia renovable|solar|eolic|tratamiento de agua'),
]

degree_or_license_led = re.compile(r'\benfermer(?:o|a|os|as)\b|\bmedic[oa]|fisioterapeut|terapeutas? ocupacional|\bfarmaceutic|ats/due|cuerpo.*titulados sanitarios|gerentes?.*(?:area|centro).*sanitari|dentist|odontolog|limpiador.*instituciones sanitarias|trabajador.*social|educador.*social|directores?.*servicios sociales|emplead.*hogar|arquitect|ingenier|psicolog|veterinari|abogad|profesor|conductor|conductores|grua torre|pala cargadora', re.I)
eoc_known_false_positive = re.compile(
    r'mecanic.*reparador.*maquinaria de construccion|'
    r'pintor.*estructuras metalicas.*cascos de buques|'
    r'pintor.*decorador.*rotulos',
    re.I,
)

def first_match(title, rules):
    for family, pattern in rules:
        if re.search(pattern, title, flags=re.I):
            return family
    return 'Sin asignación segura'

df['sector_candidate'] = df['title_normalized'].map(lambda value: first_match(value, sector_rules))
df['fp_candidate_family'] = df['sector_candidate']
mask_degree_or_license_led = df['title_normalized'].str.contains(
    degree_or_license_led,
    na=False,
)
df.loc[mask_degree_or_license_led, 'fp_candidate_family'] = 'No FP o relación insuficiente desde el título'
mask_eoc_false_positive = (
    df['fp_candidate_family'].eq('Edificación y Obra Civil')
    & df['title_normalized'].str.contains(eoc_known_false_positive, na=False)
)
df.loc[mask_eoc_false_positive, 'fp_candidate_family'] = 'No FP o relación insuficiente desde el título'
df.loc[df['sector_candidate'].eq('Sin asignación segura'), 'fp_candidate_family'] = 'Sin asignación segura'


In [4]:
import unicodedata

EOC_KNOWN_FALSE_POSITIVES = [
    'MECÁNICOS REPARADORES DE MAQUINARIA DE CONSTRUCCIÓN, MOVIMIENTO DE TIERRAS Y/O MINERÍA',
    'PINTORES DE ESTRUCTURAS METÁLICAS Y CASCOS DE BUQUES',
    'PINTORES-DECORADORES DE RÓTULOS',
]

def classified_family(raw_title):
    normalized = unicodedata.normalize('NFKD', raw_title).encode(
        'ascii', errors='ignore'
    ).decode('ascii').lower()
    matching = df.loc[df['title_normalized'].eq(normalized), 'fp_candidate_family']
    unique_families = matching.unique().tolist()
    assert len(unique_families) == 1, (raw_title, unique_families)
    return unique_families[0]

contamination_failures = []
degree_title = 'Trabajador/a social - coordinador/a de ayuda a domicilio para León'
if classified_family(degree_title) != 'No FP o relación insuficiente desde el título':
    contamination_failures.append(degree_title)
for title in EOC_KNOWN_FALSE_POSITIVES:
    if classified_family(title) == 'Edificación y Obra Civil':
        contamination_failures.append(title)
if classified_family('AUXILIARES DE ENFERMERÍA') != 'Sanidad':
    contamination_failures.append('AUXILIARES DE ENFERMERÍA')
assert not contamination_failures, contamination_failures


## Results

### 3. Volumen sectorial y volumen candidato para FP

In [5]:
window = df[df['publishedAt'] >= cutoff_6m].copy()
sector_rank = (window.groupby('sector_candidate').size().rename('offers_available_window').sort_values(ascending=False).reset_index())
fp_rank = (window.groupby('fp_candidate_family').size().rename('offers_available_window').sort_values(ascending=False).reset_index())
sector_rank['share_pct'] = (sector_rank['offers_available_window'] / len(window) * 100).round(1)
fp_rank['share_pct'] = (fp_rank['offers_available_window'] / len(window) * 100).round(1)
display(sector_rank.head(12))
display(fp_rank.head(15))

,sector_candidate,offers_available_window,share_pct
0,Sanidad,382,35.5
1,Sin asignación segura,192,17.8
2,Servicios Socioculturales y a la Comunidad,152,14.1
3,Transporte y Mantenimiento de Vehículos,85,7.9
4,Hostelería y Turismo,69,6.4
5,Edificación y Obra Civil,39,3.6
6,Comercio y Marketing,35,3.2
7,Agraria,28,2.6
8,Administración y Gestión,17,1.6
9,Instalación y Mantenimiento,14,1.3


,fp_candidate_family,offers_available_window,share_pct
0,No FP o relación insuficiente desde el título,392,36.4
1,Sin asignación segura,192,17.8
2,Servicios Socioculturales y a la Comunidad,127,11.8
3,Sanidad,88,8.2
4,Hostelería y Turismo,69,6.4
5,Edificación y Obra Civil,36,3.3
6,Comercio y Marketing,35,3.2
7,Agraria,27,2.5
8,Transporte y Mantenimiento de Vehículos,18,1.7
9,Administración y Gestión,17,1.6


### 4. Composición reciente y concentración de títulos

El corte de tres meses describe cuánto del snapshot procede del tramo reciente. Como el 94,5 % de los registros cae en ese tramo, no demuestra estabilidad temporal ni una tendencia de seis meses. Los títulos dominantes permiten auditar por qué una familia aparece arriba.

In [6]:
last3 = df[df['publishedAt'] >= cutoff_3m]
rank_available = window[~window['fp_candidate_family'].isin(['Sin asignación segura', 'No FP o relación insuficiente desde el título'])].groupby('fp_candidate_family').size().rename('offers_available_window')
rank_recent = last3[~last3['fp_candidate_family'].isin(['Sin asignación segura', 'No FP o relación insuficiente desde el título'])].groupby('fp_candidate_family').size().rename('offers_recent_3m')
recency_composition = pd.concat([rank_available, rank_recent], axis=1).fillna(0).astype(int).sort_values('offers_available_window', ascending=False).reset_index()
recency_composition['recent_3m_share_of_available_pct'] = (recency_composition['offers_recent_3m'] / recency_composition['offers_available_window'] * 100).round(1)
display(recency_composition.head(12))
top_families = recency_composition.head(6)['fp_candidate_family'].tolist()
top_titles = (window[window['fp_candidate_family'].isin(top_families)].groupby(['fp_candidate_family', 'title']).size().rename('offers').reset_index().sort_values(['fp_candidate_family', 'offers'], ascending=[True, False]).groupby('fp_candidate_family').head(8))
display(top_titles)

,fp_candidate_family,offers_available_window,offers_recent_3m,recent_3m_share_of_available_pct
0,Servicios Socioculturales y a la Comunidad,127,113,89.0
1,Sanidad,88,83,94.3
2,Hostelería y Turismo,69,64,92.8
3,Edificación y Obra Civil,36,34,94.4
4,Comercio y Marketing,35,34,97.1
5,Agraria,27,24,88.9
6,Transporte y Mantenimiento de Vehículos,18,17,94.4
7,Administración y Gestión,17,16,94.1
8,Instalación y Mantenimiento,14,14,100.0
9,Industrias Alimentarias,12,12,100.0


,fp_candidate_family,title,offers
6,Agraria,"PEONES AGRÍCOLAS, EN GENERAL",4
4,Agraria,"JARDINEROS, EN GENERAL",3
8,Agraria,"PEONES GANADEROS, EN GENERAL",3
7,Agraria,PEONES FORESTALES,2
11,Agraria,TRABAJADORES AGRÍCOLAS DE LA VID,2
12,Agraria,TRABAJADORES DE GANADO OVINO Y CAPRINO,2
0,Agraria,ASESORES AGRÍCOLAS,1
1,Agraria,Bolsa de Jefe/a de Brigada Forestal para Casti...,1
36,Comercio y Marketing,"MOZOS DE CARGA Y DESCARGA, ALMACÉN Y/O MERCADO...",8
20,Comercio y Marketing,AGENTES COMERCIALES,3


### 5. Cobertura y sensibilidad

In [7]:
coverage = pd.DataFrame([{
    'classified_sector': int((window['sector_candidate'] != 'Sin asignación segura').sum()),
    'classified_sector_pct': round((window['sector_candidate'] != 'Sin asignación segura').mean() * 100, 1),
    'fp_candidate': int((~window['fp_candidate_family'].isin(['Sin asignación segura', 'No FP o relación insuficiente desde el título'])).sum()),
    'fp_candidate_pct': round((~window['fp_candidate_family'].isin(['Sin asignación segura', 'No FP o relación insuficiente desde el título'])).mean() * 100, 1),
    'non_fp_or_insufficient': int((window['fp_candidate_family'] == 'No FP o relación insuficiente desde el título').sum()),
    'unassigned': int((window['fp_candidate_family'] == 'Sin asignación segura').sum()),
}])
display(coverage)
display(window[window['fp_candidate_family'].eq('Sin asignación segura')]['title'].value_counts().head(30).rename_axis('title').reset_index(name='offers'))

,classified_sector,classified_sector_pct,fp_candidate,fp_candidate_pct,non_fp_or_insufficient,unassigned
0,885,82.2,493,45.8,392,192


,title,offers
0,PERSONAL DE LIMPIEZA O LIMPIADORES EN GENERAL,34
1,Ofertas de empleo en la prensa regional,4
2,"ASISTENTES, ACOMPAÑANTES DE PERSONAS",3
3,Empleo privado en otras Comunidades Autónomas,3
4,"EURES: Información, Servicios y Ayudas económi...",2
5,MANTENEDORES DE EDIFICIOS,2
6,"PEONES DE OBRAS PUBLICAS, EN GENERAL",2
7,PROFESORES DE FORMACIÓN VIAL,2
8,Repartidor,2
9,TRABAJADORES DE GANADO PORCINO EN INTENSIVO,2


## Proposed five-cycle pilot

The demand signal counts title matches, not offers proven to be accessible from the cycle. The pilot must validate each official cycle-to-occupation relationship before any of these offers can become product coverage.

In [8]:
pilot_definitions = [
    ('SAN21', 'Cuidados Auxiliares de Enfermería', 'Sanidad', 'easy', r'auxiliares? de enfermeria|healthcare assistants'),
    ('HOT01M', 'Cocina y Gastronomía', 'Hostelería y Turismo', 'easy', r'cociner|pinche.*cocina'),
    ('SSC01M', 'Atención a Personas en Situación de Dependencia', 'Servicios Socioculturales y a la Comunidad', 'medium', r'asistent.*domic|ayuda a domicilio|cuidador.*(?:discapacidad|dependencia)'),
    ('EOC01M', 'Construcción', 'Edificación y Obra Civil', 'medium', r'albanil|encofrador|encargad.*obra|pintor|empapelador'),
    ('COM01M', 'Actividades Comerciales', 'Comercio y Marketing', 'ambiguous', r'dependient|cajer|reponedor|almacen|carga y descarga|carretiller|expendedor|personal.*supermercado'),
]
pilot = pd.DataFrame([{
    'program_key': key,
    'program': program,
    'family': family,
    'stratum': stratum,
    'title_match_demand_signal_available_window': int((
        window['fp_candidate_family'].eq(family)
        & window['title_normalized'].str.contains(pattern, regex=True)
    ).sum()),
    'attempt_status': 'not_started',
    'hours_spent': None,
    'discard_or_defer_reason': None,
} for key, program, family, stratum, pattern in pilot_definitions])
family_totals = window.groupby('fp_candidate_family').size()
assert all(
    row.title_match_demand_signal_available_window <= family_totals.get(row.family, 0)
    for row in pilot.itertuples()
), 'Every pilot signal must be a subset of its candidate family'
display(pilot)

pilot_metrics = {
    'completion_rate': 'completed_attempts / all_attempts',
    'discard_or_defer_rate': '(discarded_attempts + deferred_attempts) / all_attempts',
    'hours_per_completed_cycle': 'total_hours_all_attempts / completed_attempts',
}
display(pilot_metrics)

,program_key,program,family,stratum,title_match_demand_signal_available_window,attempt_status,hours_spent,discard_or_defer_reason
0,SAN21,Cuidados Auxiliares de Enfermería,Sanidad,easy,86,not_started,None,None
1,HOT01M,Cocina y Gastronomía,Hostelería y Turismo,easy,46,not_started,None,None
2,SSC01M,Atención a Personas en Situación de Dependencia,Servicios Socioculturales y a la Comunidad,medium,121,not_started,None,None
3,EOC01M,Construcción,Edificación y Obra Civil,medium,32,not_started,None,None
4,COM01M,Actividades Comerciales,Comercio y Marketing,ambiguous,25,not_started,None,None


{'completion_rate': 'completed_attempts / all_attempts',
 'discard_or_defer_rate': '(discarded_attempts + deferred_attempts) / all_attempts',
 'hours_per_completed_cycle': 'total_hours_all_attempts / completed_attempts'}

## Takeaways

- El ranking final debe interpretarse como una cola de investigación, no como una relación CNO o ciclo–ocupación aprobada.
- Las familias con volumen alto y títulos profesionales concretos son mejores candidatas para el piloto que los sectores inflados por profesiones universitarias.
- La tasa de `Sin asignación segura` es parte del resultado y limita la confianza.
- Para el piloto deben registrarse horas por intento, estado final y motivo de descarte/diferimiento.